In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../northbeam_accounts.csv")

In [3]:
df_clean = df.copy()

In [4]:
print("Original Shape:", df_clean.shape)

Original Shape: (424, 22)


In [5]:
missing_df = pd.DataFrame({
    "Missing Values": df_clean.isnull().sum(),
    "Percentage": round((df_clean.isnull().sum()/len(df_clean))*100,2)
})

missing_df = missing_df[missing_df["Missing Values"] > 0]

missing_df.sort_values(by="Percentage", ascending=False)

,Missing Values,Percentage
csat,230,54.25
segment,14,3.30
contract_end_date,3,0.71


In [6]:
duplicates = df_clean.duplicated().sum()

print("Duplicate Rows:", duplicates)

Duplicate Rows: 0


In [7]:
date_columns = [
    "signup_date",
    "contract_start_date",
    "contract_end_date",
    "last_login_date"
]

In [8]:
for col in date_columns:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        format="%d/%m/%Y",
        errors="coerce"
    )

In [9]:
df_clean[date_columns].dtypes

signup_date            datetime64[us]
contract_start_date    datetime64[us]
contract_end_date      datetime64[us]
last_login_date        datetime64[us]
dtype: object

In [10]:
text_columns = [
    "segment",
    "industry",
    "region",
    "billing_term"
]

for col in text_columns:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.strip()
    )

In [11]:
for col in text_columns:
    print(col)
    print(df_clean[col].unique())
    print()

segment
<StringArray>
['Mid-Market', 'SMB', 'Enterprise', nan]
Length: 4, dtype: str

industry
<StringArray>
[    'Logistics',   'Hospitality',        'Retail',          'SaaS',
         'Media', 'Manufacturing',       'FinServ',    'Healthcare',
  'Construction',     'Education']
Length: 10, dtype: str

region
<StringArray>
['North America', 'EMEA', 'APAC', 'LATAM']
Length: 4, dtype: str

billing_term
<StringArray>
['monthly', 'annual']
Length: 2, dtype: str



In [12]:
df_clean[df_clean["contract_value"] <= 0]

,account_id,account_name,segment,industry,region,signup_date,contract_start_date,contract_end_date,billing_term,contract_value,...,mau_m5,mau_m4,mau_m3,mau_m2,mau_m1,mau_m0,support_tickets_90d,escalations_90d,last_login_date,csat
26,ACC-1098,Atlas Capital,Enterprise,FinServ,EMEA,2024-02-24,2024-02-24,2027-03-23,annual,-4500.0,...,442,393,297,254,173,102,2,0,2026-05-23,3.8
63,ACC-1307,Northwind Health,Mid-Market,Education,North America,2022-08-03,2022-08-03,2027-06-17,annual,-4500.0,...,23,26,28,31,34,39,7,2,2026-05-30,NaN
73,ACC-1080,Lumen Group,Mid-Market,SaaS,North America,2024-06-23,2024-06-23,2026-01-11,annual,-4500.0,...,15,16,20,0,0,0,3,1,2025-10-12,3.6
109,ACC-1318,Keystone Group,Enterprise,SaaS,EMEA,2024-06-04,2024-06-04,2026-08-08,monthly,-4500.0,...,165,148,120,92,67,42,5,1,2026-05-15,3.4


In [13]:
df_clean[df_clean["seats_licensed"] <= 0]

,account_id,account_name,segment,industry,region,signup_date,contract_start_date,contract_end_date,billing_term,contract_value,...,mau_m5,mau_m4,mau_m3,mau_m2,mau_m1,mau_m0,support_tickets_90d,escalations_90d,last_login_date,csat


In [14]:
df_clean[df_clean["seats_active"] < 0]

,account_id,account_name,segment,industry,region,signup_date,contract_start_date,contract_end_date,billing_term,contract_value,...,mau_m5,mau_m4,mau_m3,mau_m2,mau_m1,mau_m0,support_tickets_90d,escalations_90d,last_login_date,csat


In [15]:
df_clean[df_clean["support_tickets_90d"] < 0]

,account_id,account_name,segment,industry,region,signup_date,contract_start_date,contract_end_date,billing_term,contract_value,...,mau_m5,mau_m4,mau_m3,mau_m2,mau_m1,mau_m0,support_tickets_90d,escalations_90d,last_login_date,csat


In [16]:
df_clean[
    (df_clean["csat"] < 1) |
    (df_clean["csat"] > 5)
]

,account_id,account_name,segment,industry,region,signup_date,contract_start_date,contract_end_date,billing_term,contract_value,...,mau_m5,mau_m4,mau_m3,mau_m2,mau_m1,mau_m0,support_tickets_90d,escalations_90d,last_login_date,csat


If active seats > licensed seats replace active seats with licensed seats

In [17]:
# Detect accounts where active seats exceed licensed seats

df_clean["data_quality_flag"] = 0

df_clean.loc[
    df_clean["seats_active"] > df_clean["seats_licensed"],
    "data_quality_flag"
] = 1


df_clean["seats_active"] = np.where(
    df_clean["seats_active"] > df_clean["seats_licensed"],

    df_clean["seats_licensed"],

    df_clean["seats_active"]
)

In [18]:
df_clean.describe()

,signup_date,contract_start_date,contract_end_date,contract_value,seats_licensed,seats_active,mau_m5,mau_m4,mau_m3,mau_m2,mau_m1,mau_m0,support_tickets_90d,escalations_90d,last_login_date,csat,data_quality_flag
count,424,424,421,4.240000e+02,424.000000,424.000000,424.000000,424.000000,424.000000,424.000000,424.000000,424.000000,424.000000,424.000000,424,194.000000,424.000000
mean,2024-06-19 02:53:12.452830,2024-06-19 02:53:12.452830,2026-11-06 06:12:49.596199,9.192647e+04,201.580189,134.674528,82.992925,81.372642,77.608491,71.823113,69.214623,65.181604,4.573113,0.747642,2026-05-10 04:55:28.301886,3.708763,0.030660
min,2022-04-25 00:00:00,2022-04-25 00:00:00,2025-07-30 00:00:00,-4.500000e+03,5.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2025-09-16 00:00:00,2.500000,0.000000
25%,2023-05-27 18:00:00,2023-05-27 18:00:00,2026-08-03 00:00:00,3.544005e+03,25.000000,16.000000,5.000000,5.000000,5.000000,3.000000,3.000000,1.000000,3.000000,0.000000,2026-05-16 00:00:00,3.100000,0.000000
50%,2024-06-30 12:00:00,2024-06-30 12:00:00,2026-11-02 00:00:00,9.726250e+03,68.000000,39.500000,18.500000,18.000000,19.000000,16.000000,15.500000,13.000000,4.000000,1.000000,2026-05-23 00:00:00,3.700000,0.000000
75%,2025-06-29 00:00:00,2025-06-29 00:00:00,2027-02-25 00:00:00,7.260744e+04,200.250000,124.000000,74.250000,69.250000,72.000000,68.250000,70.000000,60.250000,6.000000,1.000000,2026-05-28 00:00:00,4.200000,0.000000
max,2026-05-29 00:00:00,2026-05-29 00:00:00,2028-03-27 00:00:00,1.322116e+06,1488.000000,1301.000000,1206.000000,1151.000000,1147.000000,1080.000000,1069.000000,1154.000000,13.000000,6.000000,2026-06-01 00:00:00,5.000000,1.000000
std,NaN,NaN,NaN,2.158869e+05,327.647670,242.465289,169.800766,164.123873,153.878978,150.319060,148.807915,153.742770,2.629948,0.891247,NaN,0.674537,0.172599


In [19]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 424 entries, 0 to 423
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   account_id           424 non-null    str           
 1   account_name         424 non-null    str           
 2   segment              410 non-null    str           
 3   industry             424 non-null    str           
 4   region               424 non-null    str           
 5   signup_date          424 non-null    datetime64[us]
 6   contract_start_date  424 non-null    datetime64[us]
 7   contract_end_date    421 non-null    datetime64[us]
 8   billing_term         424 non-null    str           
 9   contract_value       424 non-null    float64       
 10  seats_licensed       424 non-null    int64         
 11  seats_active         424 non-null    int64         
 12  mau_m5               424 non-null    int64         
 13  mau_m4               424 non-null    int64    

In [20]:
df_clean.to_csv(
    "../northbeam_accounts_clean.csv",
    index=False
)